# Interactive Sampling Explorer

Inspect saved interactive sampling runs stored under `sampling_experiments/artifacts`.
Use the widgets below to pick a run, select a graph, and scrub through expansion steps while
visualizing node positions, active leaves, and expansion probabilities.

## Usage
1. Run `python sampling_experiments/runners/run_interactive_sampling.py ...` to generate artifacts.
2. Refresh the run dropdown (re-run the cell) if you add new runs.
3. Select a run, graph, and step to visualize predictions and metadata.
4. Scroll below the plot to inspect per-leaf logits/probabilities for the chosen step.

In [5]:
import sys
from pathlib import Path


def _find_repo_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'sampling_experiments').is_dir():
            return candidate
    raise RuntimeError('Could not locate repo root containing sampling_experiments/')

REPO_ROOT = _find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f"[notebook] Using repo root: {REPO_ROOT}")

[notebook] Using repo root: /Users/umer/Documents/dendrite_gen


In [6]:
from __future__ import annotations

import json
import pickle
from pathlib import Path

import ipywidgets as widgets
import networkx as nx
import plotly.graph_objects as go
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from sampling_experiments.interactive_methods import (
    GraphStepTrace,
    AugmentedGraphStepTrace,
)

In [7]:
ARTIFACT_ROOT = (REPO_ROOT / 'sampling_experiments' / 'artifacts')
RUN_CACHE: dict[Path, dict] = {}
TRACE_CACHE: dict[tuple[Path, int], list] = {}


def available_runs():
    if not ARTIFACT_ROOT.exists():
        return []
    runs = [p for p in ARTIFACT_ROOT.iterdir() if p.is_dir()]
    return sorted(runs, key=lambda p: p.stat().st_mtime, reverse=True)


def load_run_summary(run_dir: Path) -> dict:
    run_dir = run_dir.resolve()
    if run_dir not in RUN_CACHE:
        summary_path = run_dir / "run_summary.json"
        if not summary_path.exists():
            raise FileNotFoundError(f"run_summary.json missing in {run_dir}")
        with open(summary_path, "r") as f:
            RUN_CACHE[run_dir] = json.load(f)
    return RUN_CACHE[run_dir]


def load_graph_trace(run_dir: Path, graph_idx: int):
    key = (run_dir.resolve(), graph_idx)
    if key not in TRACE_CACHE:
        trace_path = run_dir / f"graph_{graph_idx}_trace.pkl"
        if not trace_path.exists():
            raise FileNotFoundError(f"Trace file missing: {trace_path}")
        with open(trace_path, "rb") as f:
            TRACE_CACHE[key] = pickle.load(f)
    return TRACE_CACHE[key]


def _to_numpy(tensor):
    if tensor is None:
        return None
    if hasattr(tensor, "detach"):
        return tensor.detach().cpu().numpy()
    return np.asarray(tensor)


def plot_graph_step(step_trace: GraphStepTrace, *, show_nodes: bool = True, show_edges: bool = True):
    pos = _to_numpy(step_trace.node_positions)
    if pos is None or pos.shape[0] == 0:
        raise ValueError("Step trace has no node positions")
    num_nodes = pos.shape[0]
    if pos.shape[1] < 3:
        pos3 = np.zeros((num_nodes, 3), dtype=float)
        pos3[:, :pos.shape[1]] = pos
    else:
        pos3 = pos[:, :3]

    edges = step_trace.edges or []
    leaf_ids = step_trace.leaf_ids or []
    logits = step_trace.expansion_logits or []
    probs = step_trace.expansion_probs or []
    leaf_info = {}
    for idx, leaf in enumerate(leaf_ids):
        info = {}
        if idx < len(logits) and logits[idx] is not None:
            info["logit"] = logits[idx]
        if idx < len(probs) and probs[idx] is not None:
            info["prob"] = probs[idx]
        leaf_info[leaf] = info

    node_colors = np.array(["#4c72b0"] * num_nodes)
    for leaf in leaf_ids:
        if 0 <= leaf < num_nodes:
            node_colors[leaf] = "#dd8452"

    hover_text = []
    for node_idx in range(num_nodes):
        coords = pos3[node_idx]
        label = [f"Node {node_idx}"]
        label.append(f"x={coords[0]:.3f}, y={coords[1]:.3f}, z={coords[2]:.3f}")
        info = leaf_info.get(node_idx)
        if info is not None:
            label.append("Leaf: yes")
            prob = info.get('prob')
            logit = info.get('logit')
            if prob is not None:
                label.append(f"Expansion prob: {prob:.3f}")
            if logit is not None:
                label.append(f"Expansion logit: {logit:.3f}")
        else:
            label.append("Leaf: no")
        hover_text.append("<br>".join(label))

    fig = go.Figure()

    if show_edges and edges:
        for u, v in edges:
            pts = pos3[[u, v]]
            fig.add_trace(
                go.Scatter3d(
                    x=pts[:, 0],
                    y=pts[:, 1],
                    z=pts[:, 2],
                    mode="lines",
                    line=dict(color="lightgray", width=3),
                    hoverinfo="skip",
                    showlegend=False,
                    name="edges",
                )
            )

    if show_nodes:
        fig.add_trace(
            go.Scatter3d(
                x=pos3[:, 0],
                y=pos3[:, 1],
                z=pos3[:, 2],
                mode="markers",
                marker=dict(
                    size=6,
                    color=node_colors,
                    line=dict(width=0.6, color="black"),
                ),
                hovertext=hover_text,
                hoverinfo="text",
                name="nodes",
            )
        )

    fig.update_layout(
        title=f"Step {step_trace.step_idx} | nodes={num_nodes} | leaves={len(leaf_ids)}",
        scene=dict(
            xaxis=dict(title="x", showgrid=True, zeroline=False),
            yaxis=dict(title="y", showgrid=True, zeroline=False),
            zaxis=dict(title="z", showgrid=True, zeroline=False),
            aspectmode="data",
        ),
        margin=dict(l=0, r=0, t=40, b=0),
        showlegend=False,
    )
    return fig

def leaf_dataframe(step_trace: GraphStepTrace) -> pd.DataFrame:
    leaf_ids = step_trace.leaf_ids or []
    rows = []
    probs = step_trace.expansion_probs or []
    logits = step_trace.expansion_logits or []
    global_ids = []
    if step_trace.extras and step_trace.extras.get("leaf_global_ids"):
        global_ids = step_trace.extras["leaf_global_ids"]
    for idx, leaf in enumerate(leaf_ids):
        rows.append(
            {
                "leaf_local": leaf,
                "leaf_global": global_ids[idx] if idx < len(global_ids) else None,
                "logit": logits[idx] if idx < len(logits) else None,
                "prob": probs[idx] if idx < len(probs) else None,
            }
        )
    return pd.DataFrame(rows)


def describe_step(step_trace: GraphStepTrace) -> str:
    remaining = step_trace.remaining_capacity
    expanded = None
    if step_trace.extras and step_trace.extras.get("expanded_leaf_ids"):
        expanded = step_trace.extras["expanded_leaf_ids"]
    parts = [f"Remaining capacity: {remaining}"]
    if expanded:
        parts.append(f"Expanded leaves (local ids): {expanded}")
    return " | ".join(parts)

In [8]:
run_dropdown = widgets.Dropdown(description="Run:")
graph_dropdown = widgets.Dropdown(description="Graph:")
step_slider = widgets.IntSlider(description="Step:", min=0, max=0, value=0)
output = widgets.Output()

show_nodes_checkbox = widgets.Checkbox(value=True, description="Show nodes")
show_edges_checkbox = widgets.Checkbox(value=True, description="Show edges")


def refresh_runs():
    runs = available_runs()
    options = [(p.name, p.name) for p in runs]
    if not options:
        run_dropdown.options = [("<no runs>", None)]
        run_dropdown.value = None
        graph_dropdown.options = []
        step_slider.max = 0
        return
    run_dropdown.options = options
    run_dropdown.value = options[0][1]


def update_graph_options():
    run_name = run_dropdown.value
    if run_name is None:
        graph_dropdown.options = []
        step_slider.max = 0
        return
    run_dir = ARTIFACT_ROOT / run_name
    summary = load_run_summary(run_dir)
    graph_meta = summary.get("graphs", [])
    if not graph_meta:
        graph_dropdown.options = []
        step_slider.max = 0
        return
    graph_options = [
        (f"#{g['graph_idx']} | nodes={g['num_nodes']} | steps={g['num_traces']}", g["graph_idx"] )
        for g in graph_meta
    ]
    graph_dropdown.options = graph_options
    graph_dropdown.value = graph_options[0][1]
    step_slider.max = graph_meta[0]["num_traces"] - 1
    step_slider.value = 0


def update_step_slider():
    run_name = run_dropdown.value
    graph_idx = graph_dropdown.value
    if run_name is None or graph_idx is None:
        step_slider.max = 0
        return
    run_dir = ARTIFACT_ROOT / run_name
    traces = load_graph_trace(run_dir, graph_idx)
    step_slider.max = max(len(traces) - 1, 0)
    if step_slider.value > step_slider.max:
        step_slider.value = step_slider.max


def render_current_step(*args):
    output.clear_output(wait=True)
    run_name = run_dropdown.value
    graph_idx = graph_dropdown.value
    if run_name is None or graph_idx is None:
        with output:
            display(Markdown("**No run selected.**"))
        return
    run_dir = ARTIFACT_ROOT / run_name
    traces = load_graph_trace(run_dir, graph_idx)
    if not traces:
        with output:
            display(Markdown("**Trace is empty.**"))
        return
    step_idx = min(step_slider.value, len(traces) - 1)
    trace = traces[step_idx]
    fig = plot_graph_step(
        trace,
        show_nodes=show_nodes_checkbox.value,
        show_edges=show_edges_checkbox.value,
    )
    df = leaf_dataframe(trace)
    desc = describe_step(trace)
    with output:
        display(Markdown(f"### Run `{run_name}` — Graph {graph_idx} — Step {step_idx}"))
        display(Markdown(desc))
        display(fig)
        if not df.empty:
            display(df)
        else:
            display(Markdown("_No active leaves at this step._"))


run_dropdown.observe(lambda change: (update_graph_options(), update_step_slider(), render_current_step()), names="value")
graph_dropdown.observe(lambda change: (update_step_slider(), render_current_step()), names="value")
step_slider.observe(lambda change: render_current_step(), names="value")
show_nodes_checkbox.observe(lambda change: render_current_step(), names="value")
show_edges_checkbox.observe(lambda change: render_current_step(), names="value")

refresh_runs()
update_graph_options()
update_step_slider()
render_current_step()

try:
    interactive_sampling_widget.close()
except NameError:
    interactive_sampling_widget = None
except Exception:
    pass

interactive_sampling_widget = widgets.VBox([
    widgets.HBox([run_dropdown, graph_dropdown]),
    widgets.HBox([show_nodes_checkbox, show_edges_checkbox]),
    step_slider,
    output,
])

display(interactive_sampling_widget)